# NVIDIA Active Speaker Detection - Python Notebook

This notebook demonstrates how to use the NVIDIA Active Speaker Detection service with Python. The service identifies which faces in a video are actively speaking by combining video, audio, and diarization data.

## Overview

The service processes video, audio, and speaker diarization data via bi-directional gRPC streaming and returns per-frame speaker detection results:

- **Service Integration**: Connect to NVIDIA Active Speaker Detection services
- **Multi-stream Input**: Send video, audio, and diarization data in interleaved chunks
- **Per-frame Results**: Get bounding boxes, speaking status, and confidence for each detected face
- **Video Output**: Generate an annotated video with speaker bounding boxes overlaid

## Requirements

- **Input Video**: Streamable MP4 file with H.264 video codec
- **Input Audio**: Separate WAV, MP3, or OPUS file (or use `skip_audio` to use audio embedded in the video; embedded stream codec is configurable, default OPUS)
- **Diarization**: JSON file with word-level speaker diarization. The client auto-detects **NVIDIA RIVA** diarized ASR output (`results[].alternatives[].words[]`) and the **sample** format (top-level `words` array) via `load_diarization` in `diarization.py`.
- **Output**: Annotated MP4 video with a three-color bounding box scheme: green (speaking), blue (not speaking, audio assigned), red (tracked only)
- **Service**: Access to a running NVIDIA Active Speaker Detection service instance

## Installation

### Requirements:
- Python 3.10 or later
- pip package manager
- Dependencies from `../requirements.txt` (gRPC, OpenCV, and related packages)

Run the cell below to install all required packages:

In [ ]:
%pip install -r ../requirements.txt

## Service Configuration

Configure the connection to your NVIDIA Active Speaker Detection service. The service can be running on your machine or on a remote server accessible from your environment.

In [ ]:
import os
import sys
import pathlib

SCRIPT_PATH = str(pathlib.Path().resolve())
print(f"Notebook path: {SCRIPT_PATH}")

sys.path.append(os.path.join(SCRIPT_PATH, "../scripts"))
sys.path.append(os.path.join(SCRIPT_PATH, "../interfaces"))

# Service connection configuration
SERVICE_HOST = "localhost"
SERVICE_PORT = 8001
SERVICE_TARGET = f"{SERVICE_HOST}:{SERVICE_PORT}"

print(f"Service target configured: {SERVICE_TARGET}")
print(f"Python paths configured for Active Speaker Detection modules")

In [ ]:
import time
from typing import Iterator

import cv2
import grpc
from IPython.display import HTML, display

from constants import (
    AUDIO_ENCODING_CONFIGS,
    VIDEO_CODEC_CONFIGS,
    DATA_CHUNK_SIZE,
    DIARIZATION_WORDS_BATCH_SIZE,
)
from diarization import load_diarization

from nvidia.ai4m.activespeakerdetection.v1 import activespeakerdetection_pb2
from nvidia.ai4m.activespeakerdetection.v1 import activespeakerdetection_pb2_grpc
from nvidia.ai4m.audio.v1 import audio_pb2
from nvidia.ai4m.video.v1 import video_pb2

print("All libraries imported successfully!")

## Helper Functions

Define functions for streaming data to and processing results from the Active Speaker Detection service.

### Function Overview

The Active Speaker Detection service uses bi-directional gRPC streaming:

1. **Request Generator**: Streams configuration, then interleaves video, audio, and diarization chunks
2. **Response Processor**: Collects per-frame speaker detections keyed by server ``frame_id`` (bboxes, speaking status, face and diarized speaker IDs)
3. **Video Writer**: Overlays speaker bounding boxes on the original video frames

In [ ]:
def read_file_chunks(filepath: str, chunk_size: int) -> Iterator[bytes]:
    """Yield successive byte chunks from a file."""
    with open(filepath, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            yield chunk


def generate_asd_requests(
    video_path: str,
    audio_path: str | None = None,
    diarization_path: str | None = None,
    audio_codec: str = "wav",
    skip_audio: bool = False,
    embedded_audio_codec: str = "opus",
) -> Iterator[activespeakerdetection_pb2.DetectActiveSpeakerRequest]:
    """Generate streaming requests for the Active Speaker Detection service.

    Sends config first, then interleaves video, audio, and diarization chunks.

    Args:
        video_path: Path to input MP4 video file.
        audio_path: Path to input WAV/MP3 audio file (ignored when skip_audio=True).
        diarization_path: Path to diarization JSON.
            Parsed with ``load_diarization``: supports RIVA ASR diarized JSON and the sample
            top-level ``words`` format.
        audio_codec: Audio codec name: ``"wav"``, ``"mp3"``, or ``"opus"``.
        skip_audio: If True, use audio embedded in the video instead of a separate file.
        embedded_audio_codec: Codec of audio embedded in video (used when skip_audio=True).

    Yields:
        DetectActiveSpeakerRequest messages.
    """
    # 1. Build and send config
    video_config = video_pb2.VideoConfig(codec=VIDEO_CODEC_CONFIGS["h264"])
    detection_config = activespeakerdetection_pb2.ActiveSpeakerDetectionConfig(
        input_video_config=video_config,
    )

    if not skip_audio:
        detection_config.input_audio_config.CopyFrom(
            audio_pb2.AudioConfig(encoding=AUDIO_ENCODING_CONFIGS[audio_codec])
        )
        detection_config.audio_source_config = (
            activespeakerdetection_pb2.AUDIO_SOURCE_CONFIG_SEPARATE_STREAM
        )
        print("Audio mode: separate stream")
    else:
        detection_config.input_audio_config.CopyFrom(
            audio_pb2.AudioConfig(encoding=AUDIO_ENCODING_CONFIGS[embedded_audio_codec])
        )
        detection_config.audio_source_config = (
            activespeakerdetection_pb2.AUDIO_SOURCE_CONFIG_EMBEDDED_IN_VIDEO
        )
        print(f"Audio mode: embedded in video (codec={embedded_audio_codec})")

    yield activespeakerdetection_pb2.DetectActiveSpeakerRequest(config=detection_config)
    print("Configuration sent")

    # 2. Load diarization (if not skipped)
    diarization_info = None
    if diarization_path:
        diarization_info = load_diarization(diarization_path)
        print(f"Loaded diarization with {len(diarization_info.segments)} segments")
    else:
        print("Skipping diarization data")

    # 3. Stream data in interleaved chunks
    video_iter = read_file_chunks(video_path, DATA_CHUNK_SIZE)
    audio_iter = (
        read_file_chunks(audio_path, DATA_CHUNK_SIZE) if not skip_audio else iter([])
    )

    video_done = False
    audio_done = skip_audio
    diarization_done = diarization_info is None
    segment_index = 0
    chunk_counts = {"video": 0, "audio": 0, "diarization": 0}

    while not (video_done and audio_done and diarization_done):
        if not video_done:
            chunk = next(video_iter, None)
            if chunk is None:
                video_done = True
            else:
                chunk_counts["video"] += 1
                data = activespeakerdetection_pb2.ActiveSpeakerDetectionData(
                    video_data=chunk
                )
                yield activespeakerdetection_pb2.DetectActiveSpeakerRequest(data=data)

        if not audio_done:
            chunk = next(audio_iter, None)
            if chunk is None:
                audio_done = True
            else:
                chunk_counts["audio"] += 1
                data = activespeakerdetection_pb2.ActiveSpeakerDetectionData(
                    audio_data=chunk
                )
                yield activespeakerdetection_pb2.DetectActiveSpeakerRequest(data=data)

        if not diarization_done and diarization_info is not None:
            all_segments = diarization_info.segments
            total = len(all_segments)
            if segment_index >= total:
                diarization_done = True
            else:
                batch_end = min(
                    segment_index + DIARIZATION_WORDS_BATCH_SIZE, total
                )
                batch = all_segments[segment_index:batch_end]
                is_last = batch_end >= total
                segment_index = batch_end
                chunk_counts["diarization"] += 1

                batch_info = activespeakerdetection_pb2.AudioDiarizationInfo(
                    segments=batch,
                    transcript=diarization_info.transcript if is_last else "",
                )
                data = activespeakerdetection_pb2.ActiveSpeakerDetectionData(
                    diarization_info=batch_info,
                )
                yield activespeakerdetection_pb2.DetectActiveSpeakerRequest(data=data)

    print(
        f"Data sent: {chunk_counts['video']} video, "
        f"{chunk_counts['audio']} audio, "
        f"{chunk_counts['diarization']} diarization chunks"
    )


print("Request generator defined successfully!")

In [ ]:
def process_asd_responses(
    response_iter,
) -> dict[int, list[dict]]:
    """Process responses and collect per-frame speaker detections.

    Aligns with ``active_speaker_detection.py``: results are keyed by server
    ``frame_id`` (not assumed sequential). Each speaker dict uses
    ``bbox`` (x/y/width/height dict), ``diarized_speaker_id``,
    ``face_id``, ``is_speaking``, and ``confidence_score``.

    Args:
        response_iter: Iterator of DetectActiveSpeakerResponse messages.

    Returns:
        Dict mapping ``frame_id`` to a list of speaker dicts.
    """
    frame_detections: dict[int, list[dict]] = {}
    start_time = time.time()

    print("Processing responses...")

    for response in response_iter:
        if response.HasField("config"):
            print("Received configuration acknowledgment from server")
            continue

        if response.HasField("keepalive"):
            continue

        if response.HasField("active_speaker_detection_result"):
            result = response.active_speaker_detection_result
            speakers = []
            for s in result.speaker_data:
                speakers.append({
                    "diarized_speaker_id": s.diarized_speaker_id,
                    "face_id": s.face_id,
                    "is_speaking": s.is_speaking,
                    "confidence_score": s.face_detection_confidence,
                    "bbox": {
                        "x": s.speaker_bbox.x,
                        "y": s.speaker_bbox.y,
                        "width": s.speaker_bbox.width,
                        "height": s.speaker_bbox.height,
                    },
                })
            frame_detections[result.frame_id] = speakers

    num_frames = len(frame_detections)
    elapsed = time.time() - start_time
    print(f"Received detections for {num_frames} unique frame IDs in {elapsed:.1f}s")
    return frame_detections


print("Response processor defined successfully!")

## Video Output Function

Create an annotated output video with speaker bounding boxes drawn on each frame using a three-color scheme:

- **Green**: Face is actively speaking
- **Blue**: Face has an assigned audio track (diarized) but is not speaking
- **Red**: Face is tracked but has no audio track assigned

Each bounding box includes a tracking label (face ID and detection confidence score) at the top-left corner. When a diarized audio track is assigned, the diarized speaker ID is displayed at the bottom-right corner.

In [ ]:
_COLOR_UNESTABLISHED = (0, 0, 255)  # Red – no audio assigned (tracked only)
_COLOR_SPEAKING = (0, 255, 0)  # Green – currently speaking
_COLOR_NOT_SPEAKING = (255, 0, 0)  # Blue – audio assigned, not speaking


def _face_color(spk: dict) -> tuple:
    """Three-color scheme: Red=tracked, Blue=audio assigned, Green=speaking."""
    if spk.get("is_speaking"):
        return _COLOR_SPEAKING
    if spk.get("diarized_speaker_id", -1) != -1:
        return _COLOR_NOT_SPEAKING
    return _COLOR_UNESTABLISHED


def _draw_bboxes(
    frame,
    speakers: list[dict],
    fw: int,
    fh: int,
) -> None:
    """Draw speaker boxes and labels (matches ``active_speaker_detection.py``)."""
    high_res = fh >= 1440
    font_scale = 1.2 if high_res else 0.6
    font_thickness = 4 if high_res else 2
    box_thickness = 6 if high_res else 3
    label_padding = 20 if high_res else 10
    label_offset = 10 if high_res else 5
    font = cv2.FONT_HERSHEY_SIMPLEX

    for spk in speakers:
        bbox = spk["bbox"]
        tracking_id = spk["face_id"]
        confidence = spk["confidence_score"]
        audio_id = spk.get("diarized_speaker_id", -1)
        is_speaking = spk.get("is_speaking", False)

        x = max(0, min(int(bbox["x"]), fw - 1))
        y = max(0, min(int(bbox["y"]), fh - 1))
        w = min(int(bbox["width"]), fw - x)
        h = min(int(bbox["height"]), fh - y)

        color = _face_color(spk)
        text_color = (0, 0, 0) if is_speaking else (255, 255, 255)
        cv2.rectangle(frame, (x, y), (x + w, y + h), color, box_thickness)

        track_label = f"Track:{tracking_id} ({confidence:.2f})"
        (ttw, tth), _ = cv2.getTextSize(track_label, font, font_scale, font_thickness)
        tl_x1, tl_y1 = x, y - tth - label_padding
        tl_x2, tl_y2 = x + ttw, y
        if tl_y1 < 0:
            tl_y1, tl_y2 = y, y + tth + label_padding
        cv2.rectangle(frame, (tl_x1, tl_y1), (tl_x2, tl_y2), color, -1)
        text_y = (y - label_offset) if tl_y1 <= y - tth - label_padding else (tl_y2 - label_offset)
        cv2.putText(
            frame,
            track_label,
            (x, text_y),
            font,
            font_scale,
            text_color,
            font_thickness,
            cv2.LINE_AA,
        )

        if audio_id != -1:
            audio_label = f"Audio:{audio_id}"
            (atw, ath), _ = cv2.getTextSize(audio_label, font, font_scale, font_thickness)
            br_x1 = x + w - atw
            br_y1 = y + h
            br_x2 = br_x1 + atw
            br_y2 = br_y1 + ath + label_padding
            if br_y2 > fh:
                br_y2 = fh
                br_y1 = br_y2 - ath - label_padding
            cv2.rectangle(frame, (br_x1, br_y1), (br_x2, br_y2), color, -1)
            cv2.putText(
                frame,
                audio_label,
                (br_x1, br_y1 + ath + label_offset),
                font,
                font_scale,
                text_color,
                font_thickness,
                cv2.LINE_AA,
            )


def write_output_video(
    input_video_path: str,
    output_video_path: str,
    frame_detections: dict[int, list[dict]],
) -> None:
    """Read the input video, draw speaker bboxes, and write to output.

    Maps decoded frame order to server ``frame_id`` using the minimum
    ``frame_id`` in the response map (same as the reference script).

    Args:
        input_video_path: Path to the source video.
        output_video_path: Destination path for the annotated video.
        frame_detections: Mapping of ``frame_id`` to speaker dicts
            (``bbox`` dict, ``diarized_speaker_id``, ``face_id``,
            ``is_speaking``, ``confidence_score``).
    """
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open input video: {input_video_path}")

    fw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    fh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_video_path, fourcc, fps, (fw, fh))
    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Cannot open output video for writing: {output_video_path}")

    frame_id_offset = min(frame_detections.keys()) if frame_detections else 0
    print(f"Writing output video: {total_frames} frames at {fps:.1f} fps ({fw}x{fh})")

    frame_idx = 0
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            det_fid = frame_idx + frame_id_offset
            if det_fid in frame_detections:
                _draw_bboxes(frame, frame_detections[det_fid], fw, fh)

            writer.write(frame)
            frame_idx += 1
    finally:
        cap.release()
        writer.release()

    print(f"Output video written to {output_video_path} ({frame_idx} frames)")


def display_video(video_path: str, width: int = 640) -> None:
    """Display an MP4 video inline in the notebook."""
    if not os.path.exists(video_path):
        print(f"Video not found: {video_path}")
        return
    html = f"""
    <video width="{width}" controls>
        <source src="{video_path}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
    """
    display(HTML(html))


print("Video output functions defined successfully!")

## Main Detection Pipeline

Combine all helper functions into a single detection pipeline that analyzes a video and produces an annotated output.

In [ ]:
def detect_active_speakers(
    video_path: str,
    audio_path: str | None = None,
    diarization_path: str | None = None,
    output_path: str = "asd_output.mp4",
    audio_codec: str = "wav",
    skip_audio: bool = False,
    embedded_audio_codec: str = "opus",
    server_target: str = SERVICE_TARGET,
    show_video: bool = True,
) -> dict[int, list[dict]]:
    """Complete pipeline to run Active Speaker Detection.

    Args:
        video_path: Path to input MP4 video file.
        audio_path: Path to input WAV/MP3/OPUS audio file (ignored when skip_audio=True).
        diarization_path: Path to diarization JSON.
            Loaded with ``load_diarization`` (RIVA or sample format).
        output_path: Path for annotated output video.
        audio_codec: Separate-stream audio codec: ``wav``, ``mp3``, or ``opus``.
        skip_audio: If True, use audio embedded in the video instead of a separate file.
        embedded_audio_codec: Codec label for embedded audio when ``skip_audio`` is True.
        server_target: gRPC server address (host:port).
        show_video: Whether to display the output video inline.

    Returns:
        Per-frame detection results keyed by ``frame_id``.
    """
    audio_display = "(skipped - using embedded)" if skip_audio else str(audio_path)
    diarization_display = str(diarization_path or "(none)")

    print("=" * 60)
    print("ACTIVE SPEAKER DETECTION")
    print("=" * 60)
    print(f"Video         : {video_path}")
    print(f"Audio         : {audio_display}")
    print(f"Diarization   : {diarization_display}")
    print(f"Output        : {output_path}")
    print(f"Server        : {server_target}")
    print("=" * 60)
    print()

    with grpc.insecure_channel(server_target) as channel:
        stub = activespeakerdetection_pb2_grpc.ActiveSpeakerDetectionServiceStub(
            channel
        )

        request_gen = generate_asd_requests(
            video_path=video_path,
            audio_path=audio_path,
            diarization_path=diarization_path,
            audio_codec=audio_codec,
            skip_audio=skip_audio,
            embedded_audio_codec=embedded_audio_codec,
        )

        start_time = time.time()
        response_iter = stub.DetectActiveSpeaker(request_gen)
        frame_detections = process_asd_responses(response_iter)
        elapsed = time.time() - start_time

    print(f"\nInference completed in {elapsed:.2f}s")

    write_output_video(video_path, output_path, frame_detections)

    total_speakers: set[int] = set()
    speaking_frames = 0
    for speakers in frame_detections.values():
        for s in speakers:
            dsid = s["diarized_speaker_id"]
            if dsid != -1:
                total_speakers.add(dsid)
            if s["is_speaking"]:
                speaking_frames += 1

    total_frames = len(frame_detections)
    print(f"\n{'=' * 60}")
    print("DETECTION SUMMARY")
    print(f"{'=' * 60}")
    print(f"Total frames processed  : {total_frames}")
    print(f"Unique speakers detected: {len(total_speakers)}")
    print(f"Frames with speaking    : {speaking_frames}")
    print(f"{'=' * 60}")

    if show_video:
        print(f"\nOutput video: {output_path}")
        display_video(output_path)

    return frame_detections


print("Main detection function defined successfully!")

## Example Usage

Run Active Speaker Detection on the sample assets. Update the paths below to analyze your own videos.

In [ ]:
# Update these paths to your own files (diarization: sample ``words`` JSON or RIVA ``results`` JSON)
VIDEO_PATH = "../assets/sample_video_streamable.mp4"
AUDIO_PATH = "../assets/sample_audio.wav"
DIARIZATION_PATH = "../assets/sample_diarization.json"
OUTPUT_PATH = "asd_output.mp4"

# Run with all inputs (video + separate audio + diarization)
frame_detections = detect_active_speakers(
    video_path=VIDEO_PATH,
    audio_path=AUDIO_PATH,
    diarization_path=DIARIZATION_PATH,
    output_path=OUTPUT_PATH,
    server_target=SERVICE_TARGET,
)

# To use audio embedded in the video instead of a separate audio file:
# frame_detections = detect_active_speakers(
#     video_path=VIDEO_PATH,
#     output_path="asd_skip_audio_output.mp4",
#     diarization_path=DIARIZATION_PATH,
#     skip_audio=True,
#     embedded_audio_codec="opus",
#     server_target=SERVICE_TARGET,
# )

## Understanding the Output

The output video has bounding boxes drawn on each frame using a three-color scheme:

| Color | State |
|-------|-------|
| **Green** | Face is actively speaking |
| **Blue** | Face has an assigned audio track (diarized) but is not speaking |
| **Red** | Face is tracked but has no audio track assigned |

Each bounding box includes:
- **Top-left label**: `Track:<face_id> (<score>)` -- face tracking ID and face detection confidence (0.0--1.0)
- **Bottom-right label**: `Audio:<id>` -- diarized speaker ID (shown only when an audio track is assigned)

### Key Fields

| Field | Description |
|-------|-------------|
| `face_id` | Unique ID for each tracked face in the video |
| `diarized_speaker_id` | Diarized speaker ID from diarization data (-1 if unassigned) |
| `is_speaking` | Whether the face is actively speaking |
| `confidence_score` | Face detection confidence score (0.0 to 1.0) |
| `bbox` | Bounding box as a dict: `x`, `y`, `width`, `height` |

Results are stored in a dict keyed by **`frame_id`** from the service; overlay rendering aligns video frame index with `frame_id` using the minimum `frame_id` as an offset (same approach as `scripts/active_speaker_detection.py`).

## Additional Resources

- **Documentation**: [https://docs.nvidia.com/nim/maxine/active-speaker-detection/latest/index.html](https://docs.nvidia.com/nim/maxine/active-speaker-detection/latest/index.html)
- **Package README**: `../README.md`
- **Python Script Version**: `../scripts/active_speaker_detection.py`

## Notes

- Use `skip_audio=True` to use audio embedded in the video instead of a separate audio file; set `embedded_audio_codec` (for example `opus`) to match the container
- **`load_diarization`** in `../scripts/diarization.py` auto-detects **RIVA** diarized JSON and the **sample** top-level `words` JSON (see package README)
- Custom diarization formats: subclass `DiarizationParser` and optionally register in `_PARSERS` for `load_diarization`
- In **streaming mode**, provide diarization before or at the first inference request for correct behavior (see package README)
- For generating diarization data, refer to [NVIDIA Riva ASR](https://docs.nvidia.com/deeplearning/riva/user-guide/docs/asr/asr-overview.html)